### 학습목표
- 네이버 블로그 데이터 수집하기
- 네이버 카페 데이터를 가져왔던 경험을 통하여 동일한 프로세스로 블로그 글 수집하기

##### 진행 순서
1. "음식물 처리기 사용 후기" 검색 결과 블로그 링크(url) 분석하기
2. 키워드 검색결과, 날짜 변경이 가능한 링크(url) 생성하기
3. driver를 통하여 페이지 요청하기
4. 블로그 주소 수집하기 (href_list)
5. 블로그 접근하여 데이터 수집하기
6. 코드 통합 (한번의 실행으로 전체 코드 작동)

In [1]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from tqdm import tqdm
import time
import re
from urllib.parse import quote

from selenium import webdriver
import csv
import pandas as pd
from bs4 import BeautifulSoup as bs
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import datetime
import re
 
def preprocess_sentence_kr(w):
  w = w.strip()
  w = re.sub(r"[^0-9가-힣?.!,¿]+", " ", w)
  w = w.strip()
  return w

In [2]:
search_keyword = '가전'

In [3]:
# 네이버 로그인
url='https://nid.naver.com/nidlogin.login'
id='wnsghd100'
pw='Golem2418!!'

chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
# chrome_options.add_argument("headless") # headless option
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)

driver.implicitly_wait(2)

driver.execute_script(f"document.getElementsByName('id')[0].value=\'{id}\'")
driver.execute_script(f"document.getElementsByName('pw')[0].value=\'{pw}\'")

driver.find_element(by=By.XPATH,value='//*[@id="log.login"]').click()
time.sleep(1)




keyword = quote(search_keyword)
option = 999999 # page

cnts = 0

f = open(f'네이버카페_맘스홀릭베이비_크롤링_{search_keyword}.txt', 'w', encoding='utf-8')


for p in range(1, option+1):
    print('Page: ', p)
    
    try:

        url = f'https://cafe.naver.com/f-e/cafes/10094499/menus/0?viewType=L&ta=SUBJECT&page={p}&q={keyword}'
        

        driver.get(url)
        time.sleep(1)
        
        # 4. 지식인 주소 수집하기 (href_list)
        # driver.switch_to.frame('cafe_main')

        contents = driver.find_elements(By.CSS_SELECTOR, 'a.article')
        href_list = [c.get_attribute('href') for c in contents]
        title_list = [c.text for c in contents]

        print("게시글(url) 개수: ", len(href_list))
        # print("첫 게시글 url: ", href_list[0])


        # 5. 블로그 접근하여 데이터 수집하기
        # for href in tqdm(href_list):
        if len(href_list) == 0:
            print('Finished')
            break
        
        for href in tqdm(href_list):
            # print(href)
            cnts+= 1
            driver.get(href)
            time.sleep(0.5)
            
            
            # iframe
            driver.switch_to.frame('cafe_main')
            time.sleep(0.5)
            
            # content_ = driver.find_element(By.CSS_SELECTOR, 'div.se-main-container')
            content_ = driver.find_element(By.CSS_SELECTOR, 'div.article_container')
            # print(content_.text)
            
            f.write(content_.text)
            
            driver.switch_to.default_content()
    except:
        pass

f.close()
driver.quit()

print("전체 게시글 개수: ", cnts)

Page:  1
게시글(url) 개수:  15


100%|██████████| 15/15 [00:24<00:00,  1.67s/it]


Page:  2
게시글(url) 개수:  15


100%|██████████| 15/15 [00:28<00:00,  1.88s/it]


Page:  3
게시글(url) 개수:  15


 27%|██▋       | 4/15 [00:10<00:29,  2.64s/it]


Page:  4
게시글(url) 개수:  15


100%|██████████| 15/15 [00:27<00:00,  1.82s/it]


Page:  5
게시글(url) 개수:  0
Finished
전체 게시글 개수:  50


##### 7. 워드클라우드 시각화

In [4]:
from kiwipiepy import Kiwi # 형태소 분석기
from collections import Counter # 단어가 나온 횟수를 셈
from wordcloud import WordCloud # 워드클라우드 생성 도구
import matplotlib.pyplot as plt # 시각화 도구
from PIL import Image
import numpy as np

In [5]:
f = open(f'네이버카페_맘스홀릭베이비_크롤링_{search_keyword}.txt', 'r', encoding='utf-8')
review = f.read()
f.close()
print(len(review))

26793


In [6]:
# 도구 객체생성
kiwi = Kiwi()

# 토큰화
token = kiwi.tokenize(review)
#token

# form: 토큰화 결과
# tag: 품사 태그
# nnp: 고유명사; nng: 일반명사

In [7]:
# 일반명사만 추출 (NNG)
# 리스트명 = [실행문 반복문 조건]
# nng_token = [t.form for t in token if t.tag=='NNG']
nngp_token = [t.form for t in token if t.tag=='NNG' or t.tag=='NNP']

print(len(nngp_token))
#print(nngp_token)

3908


In [8]:
# 2글자 이상의 단어만 추출
nngp_token2 = [t for t in nngp_token if len(t)>=2]
nngp_token2

['검색',
 '허용',
 '게시물',
 '게시물',
 '확인',
 '가입',
 '필요',
 '카페',
 '멤버',
 '카페',
 '가입',
 '멤버',
 '맘스',
 '홀릭',
 '베이비',
 '엄마',
 '임신',
 '육아',
 '지식',
 '카페',
 '가입',
 '임신',
 '출산',
 '육아',
 '지식',
 '핫딜',
 '정보',
 '선배',
 '노하우',
 '엄마',
 '일상',
 '임신',
 '맘스',
 '홀릭',
 '베이비',
 '최근',
 '동안',
 '가입',
 '가입',
 '검색',
 '허용',
 '게시물',
 '게시물',
 '확인',
 '가입',
 '필요',
 '카페',
 '멤버',
 '카페',
 '가입',
 '멤버',
 '맘스',
 '홀릭',
 '베이비',
 '엄마',
 '임신',
 '육아',
 '지식',
 '카페',
 '가입',
 '임신',
 '출산',
 '육아',
 '지식',
 '핫딜',
 '정보',
 '선배',
 '노하우',
 '엄마',
 '일상',
 '임신',
 '맘스',
 '홀릭',
 '베이비',
 '최근',
 '동안',
 '가입',
 '가입',
 '검색',
 '허용',
 '게시물',
 '게시물',
 '확인',
 '가입',
 '필요',
 '카페',
 '멤버',
 '카페',
 '가입',
 '멤버',
 '맘스',
 '홀릭',
 '베이비',
 '엄마',
 '임신',
 '육아',
 '지식',
 '카페',
 '가입',
 '임신',
 '출산',
 '육아',
 '지식',
 '핫딜',
 '정보',
 '선배',
 '노하우',
 '엄마',
 '일상',
 '임신',
 '맘스',
 '홀릭',
 '베이비',
 '최근',
 '동안',
 '가입',
 '가입',
 '카페',
 '가입',
 '멤버',
 '맘스',
 '홀릭',
 '베이비',
 '엄마',
 '임신',
 '육아',
 '지식',
 '카페',
 '가입',
 '임신',
 '출산',
 '육아',
 '지식',
 '핫딜',
 '정보',
 '선배',
 '노하우',
 '엄마',
 '일상',
 '임신',
 '맘

In [9]:
# 명사의 개수를 세서 많이 나오는 단어를 활용하여 워드클라우드 생성하기
counter = Counter(nngp_token2)
counter

Counter({'가입': 156,
         '임신': 123,
         '맘스': 102,
         '홀릭': 102,
         '육아': 85,
         '카페': 83,
         '엄마': 79,
         '멤버': 73,
         '베이비': 66,
         '지식': 66,
         '댓글': 55,
         '출산': 46,
         '게시물': 40,
         '박스': 40,
         '정보': 35,
         '가전': 35,
         '일상': 34,
         '최근': 34,
         '이벤트': 34,
         '핫딜': 33,
         '선배': 33,
         '노하우': 33,
         '동안': 33,
         '할인': 30,
         '카톡': 28,
         '무료': 28,
         '필요': 26,
         '구매': 25,
         '축하': 24,
         '검색': 22,
         '확인': 22,
         '상담': 22,
         '증정': 21,
         '제품': 21,
         '허용': 20,
         '혜택': 20,
         '구성': 20,
         '게시': 20,
         '공유': 19,
         '오픈': 17,
         '리스트': 16,
         '신고': 16,
         '클린': 16,
         '악성': 16,
         '감지': 16,
         '설정': 16,
         '참여': 16,
         '투표': 16,
         '회원': 14,
         '선물': 13,
         '특별': 13,
         '추천': 13,
   

In [10]:
# 상위 N개 단어만 추출
top_N = counter.most_common(50)
top_N

[('가입', 156),
 ('임신', 123),
 ('맘스', 102),
 ('홀릭', 102),
 ('육아', 85),
 ('카페', 83),
 ('엄마', 79),
 ('멤버', 73),
 ('베이비', 66),
 ('지식', 66),
 ('댓글', 55),
 ('출산', 46),
 ('게시물', 40),
 ('박스', 40),
 ('정보', 35),
 ('가전', 35),
 ('일상', 34),
 ('최근', 34),
 ('이벤트', 34),
 ('핫딜', 33),
 ('선배', 33),
 ('노하우', 33),
 ('동안', 33),
 ('할인', 30),
 ('카톡', 28),
 ('무료', 28),
 ('필요', 26),
 ('구매', 25),
 ('축하', 24),
 ('검색', 22),
 ('확인', 22),
 ('상담', 22),
 ('증정', 21),
 ('제품', 21),
 ('허용', 20),
 ('혜택', 20),
 ('구성', 20),
 ('게시', 20),
 ('공유', 19),
 ('오픈', 17),
 ('리스트', 16),
 ('신고', 16),
 ('클린', 16),
 ('악성', 16),
 ('감지', 16),
 ('설정', 16),
 ('참여', 16),
 ('투표', 16),
 ('회원', 14),
 ('선물', 13)]

In [11]:
# WordCloud(): 스타일(배경,글꼴), 최대단어수, 마스크이미지 등 옵션을 설정
# generate_from_frequencies(): 미리 정의된 단어의 빈도수를 이용해서 워드클라우드 이미지를 생성

wc = WordCloud(
    font_path='C:/Windows/Fonts/LG PC.ttf', # malgunbd.ttf
    background_color='white'
    # mask=mask
        
).generate_from_frequencies(dict(top_N))

plt.imshow(wc)
plt.axis('off')
# plt.show()

plt.savefig(f'네이버카페_맘스홀릭베이비_크롤링_{search_keyword}.png')
plt.close()